## LLM parameter-efficient fine-tuning (PEFT)

In [9]:
import sys, os 
print('\n'.join(sys.path))

/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/17.0.0/lib/python3.12/site-packages

/cvmfs/soft.computecanada.ca/easybuild/python/site-packages
/cvmfs/soft.computecanada.ca/custom/python/site-packages
/opt/jupyterhub/lib/usercustomize
/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/python/3.12.4/lib/python312.zip
/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/python/3.12.4/lib/python3.12
/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/python/3.12.4/lib/python3.12/lib-dynload
/home/jxiang27/ai4fun/lib/python3.12/site-packages
/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/ipykernel/2026a/lib/python3.12/site-packages
/tmp/tmpu2imxk5x


In [2]:
import sys
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/17.0.0/lib/python3.12/site-packages')

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [4]:
import sys
print(sys.executable)

/home/jxiang27/ai4fun/bin/python


In [5]:
import subprocess
subprocess.run(["/home/jxiang27/ai4fun/bin/python", "-m", "pip", "show", "pyarrow"])

CompletedProcess(args=['/home/jxiang27/ai4fun/bin/python', '-m', 'pip', 'show', 'pyarrow'], returncode=1)

In [6]:
import pyarrow
print(pyarrow.__version__)

import datasets
print(datasets.__version__)

17.0.0
3.6.0


In [7]:
import pyarrow
print(pyarrow.__version__)
print(pyarrow.__file__)

17.0.0
/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/17.0.0/lib/python3.12/site-packages/pyarrow/__init__.py


In [10]:
results_path = os.path.join('results', '10_LLM_fine-tuning')

if not os.path.exists(results_path):
    os.makedirs(results_path) 

In [11]:
# Load dataset
dataset = load_dataset("imdb")
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# Tokenize the dataset
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)
tokenized_datasets = dataset.map(tokenize, batched=True)

In [13]:
from transformers import TrainingArguments, Trainer
import evaluate

# Training arguments
training_args = TrainingArguments(
    output_dir=results_path,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)
# Load metric
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle().select(range(5000)),
    eval_dataset=tokenized_datasets["test"].shuffle().select(range(1000)),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.470394,0.847000
2,0.272973,0.261129,0.912000
3,0.272973,0.286334,0.922000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=939, training_loss=0.1974181682030106, metrics={'train_runtime': 538.4192, 'train_samples_per_second': 27.859, 'train_steps_per_second': 1.744, 'total_flos': 3946665830400000.0, 'train_loss': 0.1974181682030106, 'epoch': 3.0})

In [15]:
trainer.save_model(os.path.join(results_path, 'fine_tuned_bert'))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
from peft import LoraConfig, get_peft_model

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
# Wrap the base model with LoRA
lora_model = get_peft_model(model, lora_config)

In [17]:
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle().select(range(5000)),
    eval_dataset=tokenized_datasets["test"].shuffle().select(range(1000)),
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.282338,0.903000
2,0.229003,0.268898,0.903000
3,0.229003,0.266334,0.902000


TrainOutput(global_step=939, training_loss=0.21608584630476654, metrics={'train_runtime': 429.7286, 'train_samples_per_second': 34.906, 'train_steps_per_second': 2.185, 'total_flos': 3960326246400000.0, 'train_loss': 0.21608584630476654, 'epoch': 3.0})

In [18]:
lora_model.save_pretrained(os.path.join(results_path, 'lora_fine_tuned_bert'))